# 01. CIFAR-10 Data Exploration & Diffusion Transforms

This notebook demonstrates dataset loading, pixel normalization to $[-1, 1]$, class-specific filtering, and forward diffusion perturbations.

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from src.data.dataset import get_dataloaders, CIFAR10_CLASSES
from src.data.transforms import unnormalize_to_zero_one
from src.sde import get_sde
from src.diffusion.forward_process import forward_diffuse_sde

# Load dataloader
train_loader, test_loader = get_dataloaders(batch_size=16, subset_size=1000)
batch, labels = next(iter(train_loader))

print(f"Batch Shape: {batch.shape}, Range: [{batch.min():.2f}, {batch.max():.2f}]")
print(f"Classes in batch: {[CIFAR10_CLASSES[l] for l in labels[:8]]}")

In [ ]:
# Visualize clean images
grid = make_grid(unnormalize_to_zero_one(batch[:16]), nrow=4)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('Clean CIFAR-10 Samples')
plt.show()

In [ ]:
# Visualize forward diffusion trajectory over t in [0.0, 1.0]
sde = get_sde('vp')
x_0 = batch[:1]
timesteps = torch.linspace(0.01, 1.0, 8)

noisy_list = []
for t in timesteps:
    vec_t = torch.tensor([t])
    x_t, _, _ = forward_diffuse_sde(sde, x_0, vec_t)
    noisy_list.append(unnormalize_to_zero_one(x_t))

traj_grid = make_grid(torch.cat(noisy_list, dim=0), nrow=8)
plt.figure(figsize=(14, 3))
plt.imshow(traj_grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('VP-SDE Forward Diffusion Progression: t = 0.0 -> t = 1.0')
plt.show()